### Let's further evaluate how this does on real documents.

In [11]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True
)

FastLanguageModel.for_inference(model);

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

#### Let's take our new and improved SYSTEM_PROMPT for a spin

In [12]:
SYSTEM_PROMPT = """You are a SEARCH assistant with a Python REPL. You search documents - nothing else.

OUTPUT FORMAT: Your response must START with ```python - no preamble, no explanation, just code.

CONSTRAINT: Your training data is IRRELEVANT. You know NOTHING about this document.
- Answering without searching = WRONG
- Explaining instead of searching = WRONG
- Any text before your code block = WRONG

TOOLS:
- `context` - the document (already loaded, DO NOT redefine)
- `llm_query(question, context[start:end])` - ask sub-LLM about a chunk

WORKFLOW:
1. Write ```python with print() to search
2. STOP immediately after code block
3. Wait for output (appears in next message)
4. Search more OR give FINAL(answer)

SEARCH STRATEGY:
- If find() returns -1, try DIFFERENT keywords (not the same one)
- Try simpler terms, synonyms, related concepts
- Read surrounding text with context[idx:idx+500] to understand what you found

DO NOT:
- Explain what you're doing
- Answer from memory
- Write multiple code blocks
- Add text before ```python
- Redefine the context variable

When done searching, end with: FINAL(your evidence-based answer)"""

#### Now let's use the same prompt before but base model first, no SFT model

In [13]:
messages = [
    {
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\n---\n\nWhat are the main topics discussed in this document?"
    }
]

In [14]:
input_ids = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(input_ids, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens = True)
print(base_output)

```python
idx = context.find('topic')
if idx == -1:
    idx = context.find('Topic')
print(f'Found at: {idx}')
print(context[idx:idx+500])
```


#### Output:

```python
with print():
    print(llm_query('main topics', context[0:500]))
    print(llm_query('key points', context[0:500]))
    print(llm_query('summary', context[0:500]))
```

#### Honestly not even bad

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.1-8B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model.load_adapter("/workspace/outputs/8b/rlm_lora_v3", adapter_name = "default")
FastLanguageModel.for_inference(model)
print(model); # to prove that it is actually the new model

#### Like before, let's test a new question that remains untested for this test case:

In [ ]:
messages = [
    {
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\n---\n\nWho is the author of this text?"
    }
]

In [18]:
model.disable_adapters()

FastLanguageModel.for_inference(model);

input_ids = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(input_ids, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens = True)
print(base_output)

```python
with print():
    print(llm_query("what are the main topics discussed in this document", context[0:500]))
    print(llm_query("what are the key concepts in this document", context[0:500]))
    print(llm_query("what are the main ideas presented in this document", context[0:500]))
```


#### Example output: 

```python
with print():
    print(llm_query("what are the main topics discussed in this document", context[0:500]))
    print(llm_query("what are the key concepts in this document", context[0:500]))
    print(llm_query("what are the main ideas presented in this document", context[0:500]))
```

In [26]:
model.enable_adapters()

FastLanguageModel.for_inference(model);

input_ids = tokenizer.apply_chat_template(messages, return_tensors = "pt", add_generation_prompt = True).to("cuda")
outputs = model.generate(input_ids, max_new_tokens = 512)
base_output = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens = True)
print(base_output)

```python
idx = context.find('Topic')
if idx == -1:
    idx = context.find('topic')
print(f'Topic at: {idx}')
print(context[idx:idx+500])
```


#### Example output:

```python
idx = context.find('Topic')
if idx == -1:
    idx = context.find('topic')
print(f'Topic at: {idx}')
print(context[idx:idx+500])
```

#### Let's export this as a GGUF and then create an Ollama model to use in rlm.py. RESTART KERNAL THEN RUN THIS

In [2]:
from unsloth import FastLanguageModel                                     

model, tokenizer = FastLanguageModel.from_pretrained(
  model_name = "/workspace/outputs/8b/rlm_lora_v3",
  max_seq_length = 1024,
  load_in_4bit = True,
)

model.save_pretrained_merged(
  "/workspace/outputs/8b/rlm-8b-sft-merged",
  tokenizer
)

==((====))==  Unsloth 2026.1.4: Fast Llama patching. Transformers: 4.57.1. vLLM: 0.11.2.
   \\   /|    NVIDIA GeForce RTX 5080 Laptop GPU. Num GPUs = 1. Max memory: 15.92 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Found HuggingFace hub cache directory: /workspace/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|      | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|█| 4/4 [02:21<00:00, 35.27s/


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|███| 4/4 [10:54<00:00, 163.75s/it]


Unsloth: Merge process complete. Saved to `/workspace/outputs/8b/rlm-8b-sft-merged`
